In [1]:
import dotenv

import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterstats
import exactextract

from rasterstats import zonal_stats
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.mask import mask

from food_security import salinity_correction, water_quality

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / "OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/"

In [3]:
com_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
com_gdf = com_gdf.to_crs("EPSG:4326")

In [4]:
excel_path = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt_ERF_data/data_correlation.xlsx"

def write_excel_file(df, excel_path, sheet_name, append=False):
    # Open Excel file
    try:
        # book = load_workbook(excel_path)
        with pd.ExcelWriter(
            excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
        ) as writer:
            # excel_file.book = book
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except Exception as e:
        print(e)
        df.to_excel(excel_path, sheet_name=sheet_name, index=False)

In [5]:
command_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
conversion_df = pd.read_excel(excel_path, sheet_name='command_area')

mapping = (
    command_gdf.merge(
        conversion_df[['area_map_name', 'area_name']],
        left_on='OBJECTID',
        right_on='area_map_name'
    )
    .set_index('Name')['area_name']
)

com_gdf["Name"] = com_gdf["Name"].map(mapping)

In [6]:
livestock_files = [
    src_dir / "glw_cattle_v3.tif",
    src_dir / "glw_chicken_v3.tif",
    src_dir / "glw_goats_v3.tif",
    src_dir / "glw_sheeps_v3.tif"
]

livestock_df = pd.DataFrame(
    {
        "area": com_gdf["Name"]
    }
)

for livestock_file in livestock_files:
    col = livestock_file.stem
    livestock_type = col.split("_")[1]
    with rasterio.open(livestock_file) as livestock_src:
        stats = exactextract.exact_extract(
            livestock_file,
            com_gdf,
            ["sum"],
            output="pandas"
        )
    livestock_df[livestock_type] = stats["sum"]

write_excel_file(livestock_df, excel_path, sheet_name="livestock")

In [7]:
drains_gdf = gpd.read_file(src_dir / 'drains_small.shp')
canals_gdf = gpd.read_file(src_dir / 'MWRI_canals_wgs84_edit28062021.shp')

com_gdf_ecrs = com_gdf.to_crs(drains_gdf.crs)
canals_gdf = canals_gdf.to_crs(drains_gdf.crs)

com_gdf_ecrs['drain_length'] = 0.0
com_gdf_ecrs['canal_length'] = 0.0

In [8]:
for i, row in com_gdf_ecrs.iterrows():
    area_polygon = row['geometry']
    
    drains = drains_gdf.intersection(area_polygon)
    drain_length = drains.length.sum()
    com_gdf_ecrs.loc[i, 'drain_length'] = drain_length

    canals = canals_gdf.intersection(area_polygon)
    canal_length = canals.length.sum()
    com_gdf_ecrs.loc[i, 'canal_length'] = canal_length

In [9]:
excel_canal_drain_df = pd.DataFrame(
    {
        "area": com_gdf_ecrs["Name"],
        "drain_length": com_gdf_ecrs["drain_length"],
        "canal_length": com_gdf_ecrs["canal_length"],
    }
)

write_excel_file(excel_canal_drain_df, excel_path, sheet_name="canal_and_drain_length")

In [10]:
precipation_files = [
    src_dir / "egypt_annual_precipitation.tif",
    src_dir / "Egypt_annual_potentialevapotransporation.tif",
]

precipation_df = pd.DataFrame(
    {
        "area": com_gdf["Name"]
    }
)

for precipation_file in precipation_files:
    col = precipation_file.stem
    precipation_type = col.split("_")[-1]
    with rasterio.open(precipation_file) as precipation_src:
        stats = exactextract.exact_extract(
            precipation_file,
            com_gdf,
            ["sum", "mean"],
            output="pandas"
        )
    precipation_df[precipation_type] = stats["mean"]

write_excel_file(precipation_df, excel_path, sheet_name="precipation")